# Graded mesh

A beam extracted from a plasma through a round aperture, accelerated across a gap and drifting
on to a plane 11.5 mm downstream. Its divergence is set where it is extracted: at the meniscus,
in the aperture and in the gap, where the potential changes over a few cells. Along the drift
it hardly changes. A uniform mesh pays for the gap's cells over the whole length; a mesh
graded along the beam axis keeps them where they matter and lets them grow downstream.

`GridParams(x_spacing=[(x, dx), ...])` gives the node spacing wanted at a few positions along
$x$, linear between them, and the nodes follow it with cells that change in size smoothly
(`patchwork.graded_nodes`). `x_through` puts nodes on the faces of the electrodes, so that
their surfaces fall on the mesh. The spacing across the beam is that of `num_gridpoints` along
$r$. The field solve, the space charge and the tracer use the physical cell sizes.

In [ ]:
import numpy as np
import plasma_plots as pp

from beamtracer import (
    BeamOptics,
    BeamOpticsConfig,
    BeamSpecies,
    ExtractionPlasma,
    FieldBCs,
    FieldBoundaryCondition,
    GridParams,
    ObjectParams,
)

BLUE, ORANGE, GREY = "C0", "C1", "0.6"
pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## The extraction

The case of the example "Mesh refinement", with a longer drift: protons from a plasma at 5 V
(Boltzmann electrons at 5 eV, the plasma's edge starting at $x = 0.6$ mm) are extracted
through the 1 mm aperture of the plasma electrode (0 V) by the extraction electrode at -2 kV,
in cylindrical $x$-$r$ geometry, and drift to $x = 12$ mm.

In [ ]:
LENGTH, RADIUS = 12e-3, 3e-3  # the domain [m]
PLANE = 11.5e-3  # where the beam is measured [m]


def plate(x0: float, x1: float, r0: float, voltage: float) -> ObjectParams:
    # an electrode around the axis from the radius r0 outwards, between x0 and x1
    return ObjectParams(
        type="aperture2d",
        axis=0,
        position=0.0,
        vertices=((x0, r0), (x0, 3.1e-3), (x1, 3.1e-3), (x1, r0)),
        bc_type="dirichlet",
        bc_value=voltage,
    )


def uniform(cells_per_mm: int) -> GridParams:
    return GridParams(
        num_gridpoints=(12 * cells_per_mm + 1, 3 * cells_per_mm + 1),
        upper_bound=(LENGTH, RADIUS),
        coordinates="cylindrical",
    )


neumann = FieldBoundaryCondition.neumann(0.0)
config = BeamOpticsConfig(
    grid=uniform(10),
    bcs_field=FieldBCs(
        x_lower=FieldBoundaryCondition.dirichlet(5.0),
        x_upper=FieldBoundaryCondition.dirichlet(-2000.0),
        y_lower=neumann,
        y_upper=neumann,
    ),
    objects={
        "plasma electrode": plate(0.5e-3, 1.0e-3, 1.0e-3, 0.0),
        "extraction electrode": plate(3.0e-3, 4.0e-3, 1.2e-3, -2000.0),
    },
    beams=[
        BeamSpecies(
            "H+",
            50.0,
            (0.0, 0.0),
            (0.0, 0.9e-3),
            mass=1.0,
            energy=5.0,
            transverse_temperature=0.5,
            num_particles=100,
        )
    ],  # fmt: skip
    plasma=ExtractionPlasma(5.0, 5.0, 0.6e-3),
    iterations=60,
    tolerance=1e-5,
    relaxation=0.8,
    diagnostic_planes=[PLANE],
    save_trajectories=4,
)

## The graded grid

0.05 mm cells along $x$ from the plasma through the gap to the end of the extraction
electrode at 4.5 mm, growing to 0.4 mm by 7 mm and staying there along the drift; 0.05 mm
cells across the beam. Nodes on the four faces of the two electrodes.

In [ ]:
FACES = (0.5e-3, 1.0e-3, 3.0e-3, 4.0e-3)  # the electrodes' faces along x [m]
SPACING = [(0.0, 0.05e-3), (4.5e-3, 0.05e-3), (7.0e-3, 0.4e-3)]  # (x, dx) [m]
graded = GridParams(
    num_gridpoints=(2, 61),  # along x set by x_spacing; 0.05 mm across
    upper_bound=(LENGTH, RADIUS),
    coordinates="cylindrical",
    x_spacing=SPACING,
    x_through=FACES,
)
nodes = graded.x_nodes
print(f"{nodes.size} nodes along x, {np.prod(graded.num_gridpoints)} in all")
print(
    f"cells from {np.diff(nodes).min() * 1e3:.3f} to {np.diff(nodes).max() * 1e3:.3f} mm"
)
print(
    "largest ratio of neighbouring cells:",
    round(float(np.max(np.diff(nodes)[1:] / np.diff(nodes)[:-1])), 3),
)

with pp.figure(title="Cell size along the beam axis") as fig:
    ax = fig[0]
    ax.step(nodes[:-1] * 1e3, np.diff(nodes) * 1e3, where="post", color=ORANGE,
            label="graded")  # fmt: skip
    ax.plot(*np.array(SPACING).T * 1e3, "o", color="k", label="x_spacing")
    for x in FACES:
        ax.axvline(x * 1e3, color=GREY, lw=1)
    ax.set_xlabel("$x$ [mm]  (grey: the electrodes' faces)")
    ax.set_ylabel("cell size [mm]")
    ax.legend()

## The potential and the beam

The self-consistent beam on the graded mesh: the potential, the trajectories, and every
second mesh line along $x$.

In [ ]:
model = BeamOptics(config.replace(grid=graded)).run()
ds = model.dataset()
with pp.figure(title="Potential and trajectories on the graded mesh") as fig:
    ax = fig[0]
    mesh = ax.pcolormesh(
        ds.x * 1e3, ds.r * 1e3, ds.potential.T, cmap="RdBu_r", shading="auto"
    )
    ax.figure.colorbar(mesh, ax=ax, label=r"$\phi$ [V]")
    for x in nodes[::2]:
        ax.axvline(x * 1e3, color=GREY, lw=0.3)
    for path in model.trajectories:
        ax.plot(path[:, 0] * 1e3, path[:, 1] * 1e3, color="k", lw=0.6)
    for electrode in config.objects.values():
        x, r = np.array(electrode.vertices + electrode.vertices[:1]).T * 1e3
        ax.plot(x, r, color="k", lw=2)
    ax.axvline(PLANE * 1e3, color=ORANGE, ls="--", lw=1.5)
    ax.set_xlabel("$x$ [mm]")
    ax.set_ylabel("$r$ [mm]")

## Against uniform meshes

The rms divergence of the beam at $x = 11.5$ mm on uniform meshes with 0.1, 0.05 and
0.025 mm cells, and on the graded mesh. The finest mesh is the reference.

In [ ]:
def divergence(grid: GridParams) -> tuple[int, float]:
    run = BeamOptics(config.replace(grid=grid, save_trajectories=0)).run()
    return int(np.prod(grid.num_gridpoints)), run.beam_moments(PLANE).rms_angle * 1e3


uniform_points = [divergence(uniform(n)) for n in (10, 20, 40)]
graded_point = divergence(graded)
reference = uniform_points[-1][1]
for label, (count, angle) in zip(
    ("0.1 mm", "0.05 mm", "0.025 mm", "graded"),
    [*uniform_points, graded_point],
    strict=True,
):
    print(f"{label:9s} {count:6d} nodes: {angle:.3f} mrad ({angle - reference:+.3f})")

In [ ]:
with pp.figure(title="Divergence against the nodes") as fig:
    ax = fig[0]
    counts, angles = np.array(uniform_points).T
    ax.plot(counts, angles, "o-", color=GREY, label="uniform cells")
    ax.plot(*graded_point, "s", color=ORANGE, markersize=10, label="graded")
    ax.axhline(reference, color="k", ls="--", lw=1, label="0.025 mm cells")
    ax.set_xscale("log")
    ax.set_xlabel("nodes")
    ax.set_ylabel(r"rms divergence [mrad]")
    ax.legend()

The graded mesh gives the divergence of the uniform 0.05 mm mesh, within about
0.14 mrad of the reference, with half of its nodes; the uniform 0.1 mm mesh is off by about
0.4 mrad. As with a refined box (the example "Mesh refinement"), the cells along the drift
hardly matter. A graded axis suits structure that lies across the whole beam at a fixed
position, such as the gaps of an extraction system; refined boxes suit structure that is
local or moves, and the two can be combined (`refinement` on a graded grid).